# DSA 2026 S2 Assignment

**Member ID:** [INSERT]

**Date:** [INSERT]

---

### How to use this template

The cells below are a skeleton, not an answer sheet. **Add as many cells as you need** — the headings are there so that the marker for each question can find your work, and so that the code cells give you somewhere to start. Nothing stops you splitting one into five, or working somewhere else entirely and pasting the result in.

Two things the brief asks for that this skeleton does not lay out for you:

- a **text cell above** each piece of code, explaining the step you are about to take; and
- a **text cell below** the output, saying what you make of it.

So expect to add text cells throughout — that commentary is a large part of what is being marked, and a notebook of bare code will not score well however good the code is. Keep each question's work under its own heading: different markers take different questions, and an answer that lives somewhere else will not be found.


## Setup

In [ ]:
import os, sys, sqlite3, zipfile
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules


def open_database(name="ctgov.db", archive="ctgov.db.zip"):
    """Locate the database and return its path, extracting the zip if that is all there is.

    Behaves the same locally and on Colab. This runs before sqlite3.connect() on purpose:
    connect() silently creates an empty database when the file is missing, so a missing
    file would otherwise show up much later as 'no such table'.
    """
    places = [os.getcwd()]
    if IN_COLAB:
        places += ["/content", "/content/drive/MyDrive", "/content/drive/MyDrive/DSA"]

    for d in places:
        db = os.path.join(d, name)
        if os.path.isfile(db):
            return db
        archive_path = os.path.join(d, archive)
        if os.path.isfile(archive_path):
            # Unpack to the session disk rather than back into Drive: writing 146 MB to
            # Drive is slow and consumes your quota, and the session copy reads faster.
            target = "/content" if IN_COLAB and d.startswith("/content/drive") else d
            print("Extracting %s to %s ..." % (archive, target))
            try:
                with zipfile.ZipFile(archive_path) as z:
                    z.extractall(target)
            except zipfile.BadZipFile:
                raise RuntimeError(
                    "%s could not be opened. The most likely cause is an interrupted\n"
                    "download - the file should be about 40 MB. Delete it, download it\n"
                    "again, and re-run this cell." % archive_path)
            db = os.path.join(target, name)
            if os.path.isfile(db):
                return db

    looked = "\n  ".join(places)
    if IN_COLAB:
        raise FileNotFoundError(
            "%s not found. Looked in:\n  %s\n\n"
            "On Colab you need to bring the database into the session. Either upload it:\n"
            "    from google.colab import files; files.upload()\n"
            "or mount your Drive and keep it in MyDrive:\n"
            "    from google.colab import drive; drive.mount('/content/drive')\n\n"
            "Uploading is fine for a single sitting; Drive survives a disconnect." % (name, looked)
        )
    raise FileNotFoundError(
        "%s not found. Looked in:\n  %s\n\n"
        "Put %s (or %s) beside this notebook, or change directory to the folder holding it."
        % (name, looked, name, archive)
    )


if tuple(int(x) for x in matplotlib.__version__.split(".")[:2]) < (3, 9):
    print("Warning: matplotlib %s. This notebook needs 3.9 or newer (the box plot below uses\n"
          "tick_labels=). Run:  pip install -U 'matplotlib>=3.9'" % matplotlib.__version__)

DB_PATH = open_database()
conn = sqlite3.connect(DB_PATH)
print("Connected to %s (%.0f MB) - %s environment."
      % (os.path.basename(DB_PATH), os.path.getsize(DB_PATH) / 1e6,
         "Colab" if IN_COLAB else "local"))

# A part-downloaded database opens without complaint and then fails much later with
# something unhelpful like "no such table". Check it here instead, while the cause
# is still obvious.
_integrity = conn.execute("PRAGMA quick_check(1)").fetchone()[0]
_n_tables = conn.execute("SELECT COUNT(*) FROM sqlite_master WHERE type='table'").fetchone()[0]
if _integrity != "ok" or _n_tables != 12:
    print("\nThis database does not look complete: %s, %d tables, expected 12.\n"
          "Delete ctgov.db and ctgov.db.zip, download again, and re-run this cell."
          % (_integrity, _n_tables))
else:
    print("%d tables, integrity check ok." % _n_tables)
import seaborn as sns

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'", conn)
print("Tables in database: %s" % tables['name'].tolist())

def log_ai(prompt, response, qtag="general", topic="api-call"):
    """Record one AI API call made from your own code, in the format
    compile_ai_logs.py expects (see the AI-use guidance).

    Use this only when your code itself calls an AI model (e.g. an API
    call inside a pipeline). For chat/agent conversations, save the
    exported conversation file into ai_logs/ directly instead.

    qtag: one of "Q1".."Q4", "multiQ", or "general".
    """
    from pathlib import Path
    from datetime import datetime

    log_dir = Path("ai_logs")
    log_dir.mkdir(exist_ok=True)
    stamp = datetime.now().strftime("%Y-%m-%d_%H%M")
    path = log_dir / f"{stamp}_{qtag}_{topic}.md"
    with open(path, "a", encoding="utf-8") as f:
        f.write(f"## Prompt\n\n{prompt}\n\n## Response\n\n{response}\n\n---\n\n")
    return path
   

def word_count(text):
    """Count the words in a written answer.

    Q1d, Q2d and Q3d are capped at 500 words and Q4c at 300. The brief says
    markers do not read past the limit, so it is worth checking. Paste the
    answer between triple quotes:

        word_count('''My answer goes here.''')
    """
    return len(str(text).split())


def write_manual_note(qtag, topic, note, when=None, log_dir="ai_logs"):
    """Record a conversation you could not save, named like any other log.

    A work tool blocked the export, a temporary chat expired, history was
    cleared, or you simply forgot. None of that is a breach provided you
    record it honestly: say what the conversation covered and why it is
    missing. Falsifying a record, or denying use that occurred, is not the
    same thing at all.

    The same helper is demonstrated in the Getting Started notebook.

        write_manual_note("Q1", "chatgpt-export-blocked",
                          "Talked through the stratified sample for about 15 "
                          "minutes. My work account blocks copying transcripts "
                          "out, so the exchange could not be saved.")
    """
    import os
    from datetime import date

    os.makedirs(log_dir, exist_ok=True)
    stamp = when or date.today().isoformat()          # or pass when="2026-08-14"
    path = os.path.join(log_dir, "%s_%s_%s.md" % (stamp, qtag, topic))
    with open(path, "w", encoding="utf-8") as f:
        f.write("# Manual entry - %s\n\n*%s. Written by me because the conversation "
                "itself could not be saved.*\n\n%s\n"
                % (topic.replace("-", " "), stamp, note))
    print("Wrote", path)
    return path


Connected to ctgov.db (146 MB) - local environment.
12 tables, integrity check ok.
Tables in database: ['consensus_outcomes', 'event_study_results', 'fda_drug_match', 'event_fda_features', 'filing_summary', 'filing_text', 'study_event_link', 'stock_prices', 'studies', 'event_study_enriched', 'asclepius_readouts', 'asclepius_scenario_filings']


# Question 1: Validate and improve the dataset (21 marks)

Before the CFO can rely on this dataset for capital planning, two things have to be established: whether the LLM-generated outcome classifications can be trusted, and whether the data and its text fields can be put into a form the rest of the analysis can use without leaking information it should not have.

The work below runs in order and builds shared objects — the cleaned filing text, the document-type classifier, the flagging table and the analysis table — that Questions 2, 3 and 4 draw on.

## Q1a: Evaluate LLM classifications (7 marks)

The dataset's outcome labels were produced by the dual-prompt consensus pipeline described in §5 of the methodology document. Before the IR team uses them to estimate market reactions, two things need to be established: whether a label describes correctly what the filing says, and whether the label means what the downstream analysis assumes it means. These are not the same question, and the evaluation below separates them.

The brief asks for two analyses, and they answer different halves of that:

1. **A stratified sample read against the filings.** Five classifications from each of the six documented categories (n = 30), drawn with my member ID as the seed. Each is read against the sentences in the original 8-K that mention the classified drug, to judge whether the label is accurate and whether the boundaries between adjacent categories are drawn the same way from one filing to the next.
2. **Every directional label compared to the sign of the abnormal return.** Where the two disagree, the disagreement is investigated to establish whether it reflects a classification error, a market-expectations effect, or something else.

Throughout, findings are read back against the methodology document, because several of the patterns below trace directly to design choices recorded there.

Two things worth noting before starting, both visible in the cell above and neither described in the six-category documentation:

- **The table holds 13 distinct outcome values, not 6.** The 45 records the pipeline could not resolve are stored with composite labels such as `positive/negative` and `mixed-positive/mixed-negative`. Any code that assumes six categories will silently mishandle them.
- **The `statistics` column is empty in all 4,539 rows, and `evidence` in all but those 45.** The methodology explains why: the extracted statistics shaped the classifications but were never written to the student database. The practical consequence is that the quote-first grounding that the prompts were built around cannot be audited from this database at all — every check below has to go back to `filing_text`.

In [ ]:
# ---------------------------------------------------------------------------
# Q1 shared setup: load the tables Question 1 works with, and define the two
# text helpers (cleaning + filing-type classification) that Q1a, Q1b and Q1c
# all depend on. Everything downstream in Question 1 is built from these.
# ---------------------------------------------------------------------------
import re, json, textwrap
from IPython.display import display

MEMBER_ID = 40100          # used as the random seed throughout, per the brief

consensus  = pd.read_sql("SELECT * FROM consensus_outcomes", conn)
event_res  = pd.read_sql("SELECT * FROM event_study_results", conn)
filing_sum = pd.read_sql("SELECT * FROM filing_summary", conn)
filing_txt = pd.read_sql("SELECT accession, filing_text FROM filing_text", conn)

print("consensus_outcomes  %5d rows, %d filings, %d tickers"
      % (len(consensus), consensus.accession.nunique(), consensus.ticker.nunique()))
print("event_study_results %5d rows, %d filings" % (len(event_res), event_res.accession.nunique()))
print("filing_summary      %5d rows" % len(filing_sum))
print("filing_text         %5d filings" % len(filing_txt))

# The documented six categories, and what the table actually contains.
SIX = ["positive", "negative", "mixed-positive", "mixed-negative", "inconclusive", "no_result"]
counts = consensus.outcome.value_counts()
print("\nOutcome values present in consensus_outcomes:")
display(counts.to_frame("n").assign(documented=lambda d: d.index.isin(SIX)))

# The evidence and statistics columns the methodology describes.
print("evidence blank in %d of %d rows; statistics blank in %d of %d rows"
      % ((consensus.evidence.fillna("").str.strip() == "").sum(), len(consensus),
         (consensus.statistics.fillna("").str.strip() == "").sum(), len(consensus)))
print("\nNon-blank evidence, first two rows:")
for _, r in consensus[consensus.evidence.fillna("").str.strip() != ""].head(2).iterrows():
    print("  %s  %-14s  outcome=%-30s evidence=%s"
          % (r.accession, r.ticker, r.outcome, r.evidence))

In [ ]:
# ---------------------------------------------------------------------------
# Two helpers used from here to the end of Question 1.
#
# clean_filing_text()  strips the material that is identical across filings and
#                      carries no information about the trial: the SEC cover
#                      page, the safe-harbour block, the exhibit index and the
#                      signature page. It also repairs the character-level
#                      damage left by the programmatic extraction (smart quotes,
#                      bullet glyphs, and the spaced hyphens that slide decks
#                      produce, e.g. "well - tolerated").
#
# classify_filing()    labels each filing by what kind of document it is, and
#                      counts how often it refers back to an earlier disclosure.
#                      This matters because an 8-K that *announces* a result and
#                      an 8-K that *recaps* one inside a quarterly report are
#                      very different events, even when the sentence describing
#                      the trial is word-for-word the same.
# ---------------------------------------------------------------------------
MOJIBAKE = {"’": "'", "‘": "'", "“": '"', "”": '"',
            "–": "-", "—": "-", "−": "-", "′": "'",
            "▪": " ", "●": " ", "•": " ", "·": " ",
            "Ÿ": " ", " ": " ", "�": " ",
            "☐": " ", "☒": " "}

RE_COVER = re.compile(r"^.{0,6000}?Check the appropriate box below.*?13e-4\(c\)\)", re.S | re.I)
RE_SAFE  = re.compile(
    r"(?:This (?:press release|communication|report|presentation)[^.]{0,200}(?:forward-looking)"
    r"|Forward[- ]Looking Statements?|Safe Harbor Statement|Private Securities Litigation Reform Act)"
    r".{0,12000}?(?=(?:Item \d|EX-|Exhibit \d|SIGNATURE|About [A-Z]|Contacts?:|###|$))", re.S | re.I)
RE_EXIDX = re.compile(r"Item 9\.01[^A-Za-z]{0,20}Financial Statements and Exhibits"
                      r".{0,2000}?(?=(?:SIGNATURE|EX-|Exhibit 99|$))", re.S | re.I)
RE_TAIL  = re.compile(r"(SIGNATURES?\s+Pursuant to the requirements of the Securities Exchange Act.*)$",
                      re.S | re.I)

def clean_filing_text(t):
    if not isinstance(t, str):
        return ""
    for k, v in MOJIBAKE.items():
        t = t.replace(k, v)
    t = RE_COVER.sub(" ", t, count=1)
    t = RE_SAFE.sub(" ", t)
    t = RE_EXIDX.sub(" ", t)
    t = RE_TAIL.sub(" ", t)
    t = re.sub(r"\b([A-Za-z]{2,})\s+-\s+([A-Za-z]{2,})\b", r"\1-\2", t)   # slide-deck hyphens
    t = re.sub(r"https?://\S+", " ", t)
    return re.sub(r"\s+", " ", t).strip()

# A financial-results headline, a trial-results headline, and a Reg FD deck.
RE_FIN = re.compile(
    r"(Reports?|Announces?)\b[^.]{0,90}\b(First|Second|Third|Fourth)[- ]Quarter"
    r"|(Reports?|Announces?)\b[^.]{0,70}\b(Financial|Fiscal|Full[- ]Year|Annual|Year[- ]End)\b[^.]{0,40}Results"
    r"|Quarterly (Report|Results)|Results of Operations and Financial Condition|Item 2\.02", re.I)
RE_RES = re.compile(
    r"\b(Announce[sd]?|Report(s|ed)?|Present(s|ed)?|Provide[sd]?)\b[^.]{0,150}"
    r"\b(top-?\s?line|primary endpoint|pivotal|Phase\s?[1-4I]|results? (from|of|in|for)|data (from|in|at))\b",
    re.I)
RE_DECK = re.compile(r"\b(corporate|company|investor|business)\s+(presentation|deck|update|overview)\b"
                     r"|Item 7\.01|Regulation FD Disclosure", re.I)
RE_PRIOR = re.compile(
    r"\b(previously (announced|reported|disclosed)|as (previously )?announced"
    r"|in (January|February|March|April|May|June|July|August|September|October|November|December)"
    r" 20\d\d,? (the Company |we |[A-Z][a-z]+ )?(announced|reported)|earlier this year)\b", re.I)

def classify_filing(clean_text, raw_text, head=1200, body=6000):
    """Label the document type, preferring the headline, and count restatement language.

    The headline is checked first, and a trial-results headline wins over a
    financial one, because a company announcing topline data will often mention
    a coming quarter in the same release. Falling back to the body only where
    the headline is silent stops guidance language ("we expect to report in the
    fourth quarter") from turning an announcement into a quarterly report - a
    real misclassification in an earlier version of this function, found by
    checking it against the filings read by hand for the sample below.
    """
    h, b = clean_text[:head], clean_text[:body]
    fin, res = bool(RE_FIN.search(h)), bool(RE_RES.search(h))
    if res and not fin:
        kind = "results announcement"
    elif fin or RE_FIN.search(b):
        kind = "periodic report"
    elif RE_DECK.search(b):
        kind = "deck / FD update"
    elif RE_RES.search(b):
        kind = "results announcement"
    else:
        kind = "other"
    return kind, len(RE_PRIOR.findall(raw_text))

filing_txt["clean"] = filing_txt.filing_text.map(clean_filing_text)
_kinds = [classify_filing(c, r) for c, r in zip(filing_txt.clean, filing_txt.filing_text)]
filing_txt["filing_kind"] = [k for k, _ in _kinds]
filing_txt["n_prior_refs"] = [n for _, n in _kinds]

print("Cleaning removed %.1f%% of all characters (median filing %d -> %d chars)"
      % (100 * (1 - filing_txt.clean.str.len().sum() / filing_txt.filing_text.str.len().sum()),
         filing_txt.filing_text.str.len().median(), filing_txt.clean.str.len().median()))
print()
display(filing_txt.filing_kind.value_counts().to_frame("filings"))

### Analysis 1: reading a stratified sample against the filings

Sampling is done within category rather than across the whole table because the categories are very unevenly sized — `positive` has 1,898 records and `mixed-negative` 85 — so a simple random sample of 30 would be unlikely to contain any `mixed-negative` or `inconclusive` cases at all. Five per category gives 30 records and guarantees coverage of all six documented categories, as the brief requires.

In [ ]:
# ---------------------------------------------------------------------------
# Analysis 1 of 2: stratified sample of classifications, read against the filing.
# Five records from each of the six documented categories (n=30), drawn with the
# member ID as the seed. Sampling within stratum rather than overall guarantees
# the two rarest categories (mixed-negative, inconclusive) are represented.
# ---------------------------------------------------------------------------
car_by_row = event_res.drop_duplicates(["accession", "drug_name"])[
    ["accession", "drug_name", "car_3day"]]
cons_car = consensus.merge(car_by_row, on=["accession", "drug_name"], how="left")

sample = pd.concat([cons_car[cons_car.outcome == o].sample(n=5, random_state=MEMBER_ID)
                    for o in SIX]).reset_index(drop=True)
sample = sample.merge(filing_txt[["accession", "filing_kind", "n_prior_refs"]],
                      on="accession", how="left")

print("Stratified sample: %d records, %d categories, seed=%d\n"
      % (len(sample), sample.outcome.nunique(), MEMBER_ID))
display(sample[["accession", "ticker", "filing_date", "drug_name", "phase", "outcome",
                "consensus_type", "car_3day", "filing_kind", "n_prior_refs"]])

In [ ]:
# ---------------------------------------------------------------------------
# Reading each sampled classification against its filing. The filings run to
# 20,000-90,000 characters, so rather than print them whole this pulls the
# sentences that mention the classified drug and describe a result, which is
# what the classification has to be judged against. Every judgement recorded in
# the table below was formed by reading this output for all 30 records; three
# representative cases are reproduced here.
# ---------------------------------------------------------------------------
RE_RESULT = re.compile(
    r"\b(primary endpoint|co-primary|secondary endpoint|statistically significant"
    r"|p\s?[=<>]\s?0?\.\d+|hazard ratio|odds ratio|response rate|overall survival"
    r"|progression-free|did not meet|failed to|met its|achieved|topline|top-line"
    r"|discontinu\w+|futility|terminat\w+|well[- ]tolerated|adverse events?|efficacy"
    r"|superiority|non-inferior\w*|interim analysis)\b", re.I)

def drug_tokens(name):
    name = re.sub(r"\(.*?\)", "", str(name))
    return [t.lower() for t in re.split(r"[^A-Za-z0-9]+", name) if len(t) > 3][:4]

def result_sentences(clean_text, drug, k=10, lo=40, hi=600):
    """Rank sentences by drug mention and result language; return them in reading order."""
    sents = [s for s in re.split(r"(?<=[.!?])\s+", clean_text) if lo <= len(s) <= hi]
    if not sents:
        return [clean_text[:hi]]
    toks, scored = drug_tokens(drug), []
    for i, s in enumerate(sents):
        low = s.lower()
        hit = any(t in low for t in toks)
        n_res = len(RE_RESULT.findall(s))
        score = (3 if hit else 0) + min(n_res, 3) + (2 if (hit and n_res) else 0)
        if score:
            scored.append((score, i, s))
    if not scored:
        return sents[:k]
    top = sorted(scored, key=lambda x: (-x[0], x[1]))[:k]
    return [s for _, _, s in sorted(top, key=lambda x: x[1])]

clean_by_acc = filing_txt.set_index("accession")["clean"]

def show(i, k=4):
    r = sample.loc[i]
    car = "n/a" if pd.isna(r.car_3day) else "%+.1f%%" % (100 * r.car_3day)
    print("=" * 100)
    print("[%d] %s | %s | %s | %s" % (i, r.accession, r.ticker, r.filing_date, r.drug_name))
    print("     label=%s  consensus=%s  CAR[-1,+1]=%s  document=%s  restatement phrases=%d"
          % (r.outcome, r.consensus_type, car, r.filing_kind, r.n_prior_refs))
    for s in result_sentences(clean_by_acc.get(r.accession, ""), r.drug_name, k=k):
        print(textwrap.fill(s, 98, initial_indent="   - ", subsequent_indent="     "))
    print()

for i in [6, 1, 11]:      # a clean negative; a stale positive; a boundary case
    show(i)

In [ ]:
# ---------------------------------------------------------------------------
# Before leaning on classify_filing(), check it against the 30 filings just read
# by hand. Most of Analysis 2 below turns on the announcement / not-announcement
# distinction, so it is worth knowing how often that distinction is right rather
# than assuming it.
# ---------------------------------------------------------------------------
HAND_LABELLED = [                       # index into `sample`, my reading of the document
    "results announcement", "periodic report", "results announcement", "periodic report",
    "deck / FD update", "periodic report", "results announcement", "results announcement",
    "periodic report", "periodic report", "deck / FD update", "results announcement",
    "results announcement", "periodic report", "deck / FD update", "periodic report",
    "periodic report", "periodic report", "deck / FD update", "periodic report",
    "deck / FD update", "deck / FD update", "periodic report", "periodic report",
    "deck / FD update", "other", "periodic report", "periodic report",
    "deck / FD update", "periodic report"]

check = sample[["accession", "ticker", "filing_date", "outcome", "filing_kind"]].copy()
check["read_by_hand"] = HAND_LABELLED
check["exact"] = check.filing_kind == check.read_by_hand
check["announcement_correct"] = ((check.filing_kind == "results announcement") ==
                                 (check.read_by_hand == "results announcement"))

print("Exact agreement on document type:        %d of 30" % check.exact.sum())
print("Agreement on announcement vs not:        %d of 30" % check.announcement_correct.sum())
print("\nDisagreements:")
display(check.loc[~check.exact, ["accession", "ticker", "filing_kind", "read_by_hand"]])
print("Only %d of the 30 sampled filings are the announcement that carried the news."
      % (check.read_by_hand == "results announcement").sum())

#### What reading the 30 filings showed

Each of the 30 records was read against the drug-and-result sentences of its filing. Two judgements were recorded for each: whether the **label** is a fair description of what the filing says about that drug, and whether the **filing** is the document that carried the news to the market.

| Category | Label fair on the text | Filing is the announcement | Notes |
|---|---|---|---|
| positive (5) | 5 | 2 | Two labels rest on background sections rather than new results |
| negative (5) | 5 | 2 | Three are recaps inside quarterly reports |
| mixed-positive (5) | 3 | 2 | Two sit on the wrong side of a boundary applied differently elsewhere |
| mixed-negative (5) | 5 | 0 | All five are periodic reports or investor decks |
| inconclusive (5) | 1 | 0 | Four describe trials that have not reported at all |
| no_result (5) | 5 | n/a | Correct; these are excluded from the event study |

Read as text classification, the pipeline does well: in 24 of the 30 records the label is a defensible reading of the sentences it is drawn from, and there is no case in the sample where a plainly positive result was called negative or the reverse. The problems are of three other kinds.

**(1) The label is accurate but the filing is not the event.** Only **6 of the 30** sampled filings are the announcement that carried the news: 15 are quarterly or annual reports, 8 are investor decks or Reg FD updates, and 1 is a securities purchase agreement. In the periodic reports the sentence the classification rests on is a recap of something announced weeks or months earlier. Four clear examples from the sample:

- **PFE / Xeljanz** (`0000078003-16-000091`, 3 May 2016) — a Q1 earnings 8-K. The OPAL Broaden result it describes was announced in April 2016; the filing carries six restatement phrases. Labelled `positive`.
- **IMUX / vidofludimus calcium** (`0001193805-22-001141`, 4 Aug 2022) — a Q2 results 8-K restating the June 2022 CALDOSE-1 primary-endpoint miss. Labelled `negative`; the CAR around this filing is **+41.8%**.
- **REGN / suptavumab** (`0001532176-17-000033`, 8 Nov 2017) — a Q3 report. The Phase 3 miss and the decision to discontinue development were announced in August 2017. Labelled `negative`; CAR +0.4%.
- **PFE / fordadistrogene movaparvovec** (`0000078003-24-000155`, 30 July 2024) — a Q2 report restating the June 2024 CIFFREO miss. Labelled `negative`; CAR −1.1%.

In each case the classification is right about the trial and wrong about the day. The abnormal return sitting beside it is measuring a quarterly earnings reaction, not a reaction to the trial.

**(2) The mixed-positive / mixed-negative boundary follows the company's framing, not the endpoint.** The cell below sets out the decisive case: two Tonix filings describe the same study — P201/AtEase in military-related PTSD, whose pre-specified primary endpoint did not separate from placebo — and receive opposite labels. The 2016 filing, which presents the secondary and post-hoc results first, is `mixed-positive`; the 2019 filing, which states the primary endpoint miss plainly before the retrospective analyses, is `mixed-negative`. The same pattern holds across the sample: TNXP's BESTFIT study (primary endpoint p = 0.172, several secondaries significant) is `mixed-positive`, while VCNX's pepinemab (co-primaries missed, favourable cognitive trends) and LYRA's LYR-210 (primary missed) are `mixed-negative`. What separates them is how positively the sponsor wrote it up.

**(3) `inconclusive` does not mean what it is documented to mean.** The documented definition is "results reported but insufficient to determine direction". Four of the five sampled records are filings where the trial has not reported at all: TBPH's ampreloxetine ("Phase 3, awaiting topline data"), GUTS's Revita ("data expected in Q3 2025"), NBIX's indiplon (enrolment updates) and LCTX's AST-VAC1 (a corporate deck). This follows directly from the design of prompt B3b, which the methodology states defaults to `inconclusive` wherever it cannot populate the statistics field. In practice the category is a second `no_result`, and the two overlap.

Two smaller observations. The `positive` label on **LPCN 1111** (`0001144204-16-125422`) is drawn from a background feasibility claim in an investor deck whose actual news included a Complete Response Letter for a different asset. And two records — TCRT's `Ad-IL12 / DC-IL12` and LCTX's `AST-VAC1` — name drugs whose text does not appear in the filing at all; that is checked across the whole table further below.

These 30 hand-read documents are also what `classify_filing()` was validated against in the cell above, since Analysis 2 below leans on that distinction across all 3,794 filings. It agrees with my reading on the announcement / not-announcement split in 28 of 30 cases. Its residual weakness is telling apart investor decks from quarterly reports — both of which are "not the announcement", so the distinction that matters is the more reliable one. An earlier version was materially worse: it read the guidance sentence "we expect to report top-line results in late fourth quarter 2021" as a quarterly report and misfiled a genuine Theravance topline announcement, which is why the headline now takes precedence over the body.

In [ ]:
# ---------------------------------------------------------------------------
# The boundary test. Reading the sample suggested the mixed-positive /
# mixed-negative line is drawn by the *tone the company adopts*, not by whether
# the primary endpoint was met. Two Tonix filings describe the same study
# (P201/AtEase in military-related PTSD, whose pre-specified primary endpoint
# missed) and receive opposite labels, so the two can be read side by side.
# ---------------------------------------------------------------------------
pair = consensus[consensus.accession.isin(["0001144204-16-105673", "0001387131-19-001982"])]
display(pair[["accession", "ticker", "filing_date", "drug_name", "outcome", "consensus_type"]])

for acc in ["0001144204-16-105673", "0001387131-19-001982"]:
    row = consensus[consensus.accession == acc].iloc[0]
    print("=" * 100)
    print("%s  %s  labelled: %s" % (acc, row.filing_date, row.outcome))
    for s in result_sentences(clean_by_acc[acc], "TNX-102 SL cyclobenzaprine", k=4):
        print(textwrap.fill(s, 98, initial_indent="   - ", subsequent_indent="     "))
    print()

The two filings describe the same study and the same missed primary endpoint. The 2016 filing leads with the secondary and post-hoc results and is labelled `mixed-positive`; the 2019 filing states the primary endpoint miss before the retrospective analyses and is labelled `mixed-negative`. Nothing about the trial changed between them.

This is the clearest evidence in the sample that the mixed-positive / mixed-negative boundary is not being drawn on the endpoint. It is being drawn on how the sponsor chose to present the result — which is exactly the property a market-reaction model should not inherit, because the market prices the result rather than the framing.

### Analysis 2: comparing every label to the sign of the abnormal return

The sample above can only speak for 30 records. This second analysis tests the same concerns across all 2,392 directional classifications with an event study attached, using the sign of `car_3day` as an external check on the label.

The check is deliberately weak: a `positive` label does not have to be followed by a positive return for the label to be right — good news can be already priced in, or can fall short of what the market hoped for. So a high disagreement rate is not by itself evidence of error. What makes the comparison useful is the *pattern* of disagreement: if the labels are sound and the disagreements are market effects, disagreement should not depend on what kind of document the 8-K is.

In [ ]:
# ---------------------------------------------------------------------------
# Analysis 2 of 2: compare every directional label to the sign of the abnormal
# return, across the whole dataset. 'no_result' and 'inconclusive' carry no
# directional claim and are excluded; the remaining four categories imply a
# direction the market should have moved in.
#
# Reported at record level (as classified) and at filing level (deduplicated
# via filing_summary), because 1,073 of the 4,539 records sit in multi-drug
# filings and share one CAR between them.
# ---------------------------------------------------------------------------
EXPECTED = {"positive": 1, "mixed-positive": 1, "negative": -1, "mixed-negative": -1}

def sign_table(df, label_col):
    d = df.assign(expected=df[label_col].map(EXPECTED)).dropna(subset=["expected", "car_3day"])
    d = d.assign(disagrees=np.sign(d.car_3day) != d.expected)
    t = d.groupby(label_col).agg(n=("car_3day", "size"),
                                 disagreement_rate=("disagrees", "mean"),
                                 mean_car=("car_3day", "mean"),
                                 median_car=("car_3day", "median"),
                                 sd_car=("car_3day", "std"))
    return t.reindex([c for c in EXPECTED if c in t.index]).round(3), d

rec_tbl, rec_d = sign_table(event_res, "outcome")
fil_tbl, fil_d = sign_table(filing_sum, "primary_outcome")
print("RECORD LEVEL (event_study_results, n=%d directional records)" % len(rec_d))
display(rec_tbl)
print("FILING LEVEL (filing_summary, n=%d directional filings)" % len(fil_d))
display(fil_tbl)
print("Overall record-level disagreement rate: %.1f%%" % (100 * rec_d.disagrees.mean()))

from scipy import stats
pos = fil_d.loc[fil_d.primary_outcome == "positive", "car_3day"]
neg = fil_d.loc[fil_d.primary_outcome == "negative", "car_3day"]
u, p = stats.mannwhitneyu(pos, neg)
print("\nDo the labels separate returns at all? positive (n=%d) vs negative (n=%d):"
      % (len(pos), len(neg)))
print("  Mann-Whitney U p = %.2e" % p)
print("  positive: mean %+.2f%%, one-sample t vs 0 p = %.1e" % (100 * pos.mean(), stats.ttest_1samp(pos, 0).pvalue))
print("  negative: mean %+.2f%%, one-sample t vs 0 p = %.1e" % (100 * neg.mean(), stats.ttest_1samp(neg, 0).pvalue))

fig, ax = plt.subplots(figsize=(8, 4))
order = ["negative", "mixed-negative", "mixed-positive", "positive"]
ax.boxplot([fil_d.loc[fil_d.primary_outcome == o, "car_3day"] for o in order],
           tick_labels=order, showfliers=False)
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("CAR[-1,+1]"); ax.set_title("Market reaction by LLM label (filing level)")
plt.tight_layout(); plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Investigating the disagreements. Reading the sample suggested most of them are
# not misreadings of the filing: the label describes the trial correctly, but
# the filing it is attached to is not the day the news reached the market. If
# that is right, disagreement should depend on what kind of document the 8-K is
# and on how much restatement language it contains - and it should be strongest
# where the label is most reliable in a genuine announcement.
# ---------------------------------------------------------------------------
fs = filing_sum.merge(filing_txt[["accession", "filing_kind", "n_prior_refs"]],
                      on="accession", how="left")
fs["expected"] = fs.primary_outcome.map(EXPECTED)
d = fs.dropna(subset=["expected", "car_3day"]).copy()
d["agrees"] = np.sign(d.car_3day) == d.expected
d["abs_car"] = d.car_3day.abs()

by_kind = d.groupby("filing_kind").agg(n=("agrees", "size"), sign_agreement=("agrees", "mean"),
                                       median_abs_car=("abs_car", "median"),
                                       sd_car=("car_3day", "std")).round(3)
print("Sign agreement and reaction size by document type")
display(by_kind.sort_values("sign_agreement", ascending=False))

print("Sign agreement by document type x label (cell = agreement rate, n in brackets)")
piv = d.pivot_table(index="filing_kind", columns="primary_outcome", values="agrees",
                    aggfunc=["mean", "size"])
display(piv.round(2))

d["restatement_phrases"] = pd.cut(d.n_prior_refs, [-1, 0, 2, 5, 10**6],
                                  labels=["0", "1-2", "3-5", "6+"])
print("Sign agreement falls monotonically as the filing refers back to earlier disclosures")
display(d.groupby("restatement_phrases", observed=True)
         .agg(n=("agrees", "size"), sign_agreement=("agrees", "mean"),
              median_abs_car=("abs_car", "median")).round(3))

In [ ]:
# ---------------------------------------------------------------------------
# Reading a fresh selection of disagreement cases. They are chosen stratified by
# document type rather than by size of move, for two reasons. First, a
# disagreement inside a quarterly report and one inside a genuine announcement
# are candidates for different explanations, and picking on size alone would
# have produced only the second kind. Second, car_3day is winsorised at the 1st
# and 99th percentiles: 54 records (2.1%) sit at exactly -84.2% or +94.5%, so
# ranking on the extreme tail returns ties rather than the true extremes.
# ---------------------------------------------------------------------------
cap_lo, cap_hi = event_res.car_3day.min(), event_res.car_3day.max()
print("Winsorisation caps: %d records at exactly %+.1f%%, %d at exactly %+.1f%% (%.1f%% of rows)"
      % ((event_res.car_3day == cap_lo).sum(), 100 * cap_lo,
         (event_res.car_3day == cap_hi).sum(), 100 * cap_hi,
         100 * ((event_res.car_3day == cap_lo) | (event_res.car_3day == cap_hi)).mean()))

contra = (rec_d[rec_d.disagrees]
          .merge(filing_txt[["accession", "filing_kind", "n_prior_refs"]], on="accession", how="left")
          .assign(abs_car=lambda x: x.car_3day.abs()))
contra = contra[(contra.car_3day > cap_lo) & (contra.car_3day < cap_hi)]

picks = (contra[contra.filing_kind.isin(["results announcement", "periodic report"])]
         .sort_values("abs_car", ascending=False)
         .groupby("filing_kind").head(4)
         .sort_values(["filing_kind", "abs_car"], ascending=[True, False]))
display(picks[["accession", "ticker", "event_date", "drug_name", "outcome", "consensus_type",
               "car_3day", "filing_kind", "n_prior_refs"]].round(3))

for _, r in picks.groupby("filing_kind").head(2).iterrows():
    print("=" * 100)
    print("%s | %s | %s | label=%s | CAR=%+.1f%% | document=%s | restatement phrases=%d"
          % (r.accession, r.ticker, r.drug_name, r.outcome, 100 * r.car_3day,
             r.filing_kind, r.n_prior_refs))
    for s in result_sentences(clean_by_acc.get(r.accession, ""), r.drug_name, k=4):
        print(textwrap.fill(s, 98, initial_indent="   - ", subsequent_indent="     "))
    print()

In [ ]:
# ---------------------------------------------------------------------------
# Three further checks the sample pointed at, run over the whole table:
#   (a) is the same drug in the same filing classified more than once, and do
#       those duplicates agree with each other?
#   (b) does the classified drug name appear anywhere in the filing at all?
#       (the methodology notes that drug matching between prompts used substring
#       containment, which can cross-match)
#   (c) does the provenance recorded in consensus_type predict reliability?
# ---------------------------------------------------------------------------
named = consensus.drug_name.notna() & (consensus.drug_name.fillna("").str.strip() != "")
grp = consensus[named].groupby(["accession", "drug_name"])["outcome"]
n_rows, n_distinct = grp.transform("size"), grp.transform("nunique")
dupes = pd.Series(0, index=consensus.index)
dupes[named] = np.where(n_distinct > 1, 2, np.where(n_rows > 1, 1, 0))
consensus["dup_status"] = dupes.map({0: "unique", 1: "repeated, same label",
                                     2: "repeated, CONFLICTING labels"})
print("(a) Duplicate (filing, drug) classifications")
display(consensus.dup_status.value_counts().to_frame("records"))
print("Example of a conflicting duplicate:")
display(consensus[consensus.dup_status == "repeated, CONFLICTING labels"]
        .sort_values(["accession", "drug_name"])
        [["accession", "ticker", "drug_name", "outcome", "consensus_type"]].head(6))

raw_by_acc = filing_txt.set_index("accession")["filing_text"]

def drug_present(acc, drug, outcome):
    """Search the RAW filing, not the cleaned one.

    The safe-harbour block routinely lists a company's product names, and
    clean_filing_text() removes it. Checking against the cleaned text would
    therefore report a drug as absent when all my cleaning did was delete the
    only place it was mentioned - which is a different (and much weaker) claim
    than 'this drug is not in this filing at all'.
    """
    if outcome == "no_result" or not isinstance(drug, str) or not drug.strip():
        return np.nan
    text = raw_by_acc.get(acc)
    if text is None:
        return np.nan
    toks, low = drug_tokens(drug), text.lower()
    return bool(toks) and any(t in low for t in toks)

consensus["drug_in_text"] = [drug_present(a, d, o) for a, d, o
                             in zip(consensus.accession, consensus.drug_name, consensus.outcome)]
absent = consensus.drug_in_text == False
print("\n(b) Classified drug name not found anywhere in the filing: %d of %d classifications "
      "that name a drug (%.1f%%)"
      % (absent.sum(), consensus.drug_in_text.notna().sum(),
         100 * absent.sum() / consensus.drug_in_text.notna().sum()))
display(consensus[absent].outcome.value_counts().head(6).to_frame("records"))

print("\n(c) Reliability by recorded provenance (directional, event-linked records)")
prov = rec_d.assign(agrees=~rec_d.disagrees).groupby("consensus_type").agg(
    n=("agrees", "size"), sign_agreement=("agrees", "mean"),
    median_abs_car=("car_3day", lambda s: s.abs().median())).round(3)
display(prov.sort_values("n", ascending=False))

#### What the label-versus-return comparison shows

The labels do carry real information. Filings labelled `negative` average a **−19.2%** three-day abnormal return and those labelled `positive` **+3.7%**; the two distributions are separated at p ≈ 3.5 × 10⁻³³. Whatever its defects, this is not a pipeline that has classified at random.

But sign agreement is very uneven, and in one place it is no better than a coin toss:

| Label | n (filings) | Disagreement with return sign | Mean CAR |
|---|---|---|---|
| negative | 338 | 29.0% | −19.2% |
| mixed-negative | 69 | 37.7% | −10.4% |
| positive | 1,191 | **49.0%** | +3.7% |
| mixed-positive | 158 | **67.1%** | **−6.6%** |

Two things stand out. A `positive` label predicts the direction of the move no better than a coin. And `mixed-positive` is *negative* on average: the market treats it as bad news. That matters beyond this question, because the consensus rules in §5 of the methodology resolve `positive` vs `mixed-positive` to `positive` as a "same direction" pair, and `outcome_group` in the event study tables maps `mixed-positive` into `Positive`. Both treat as favourable a category the market reliably marks down.

#### Separating the causes

Disagreement is not evidence of error on its own. The analysis below separates four distinct causes rather than attributing all of them to market efficiency.

**(a) The filing is not the disclosure event — the largest single cause.** Splitting on document type, sign agreement is **60.9%** in genuine results announcements against **46.0%** in periodic reports, and the median absolute reaction is 13.1% against 5.1%. The effect is sharpest exactly where the label should be most informative: a `negative` label agrees with the return **90%** of the time in an announcement and **53%** of the time in a quarterly report. The same gradient appears in restatement language independently of document type — agreement falls from 56.6% in filings with no backward reference to 38.1% in those with three to five, and the median move shrinks with it. This is what mis-dated events look like: as the news gets older, the label keeps describing the trial correctly while the return beside it converges on noise.

This is the defect that prompt **B1b** was designed to catch. Round 3 of the prompt selection tested a variant that flagged "whether the announcement was new rather than a restatement of previously disclosed results", and B1a was selected over it. The methodology records the choice without recording what it cost; the answer is that roughly half the directional records are attached to filings that are not the event.

**(b) The label attaches to the wrong news within the same filing.** Neurogene's November 2024 filing (`0001404644-24-000103`) is labelled `mixed-positive`, anchored on "the favorable safety and efficacy data for NGN-401 at the 1E15 vg dose". The filing's actual news, two sentences later, is a treatment-related serious adverse event at the higher dose and a halt to further dosing there. The market fell **67.8%**. Sellas (`0001390478-22-000030`) is labelled `positive` from a top-line result announced four days earlier, while the filing itself announces a change to the Phase 3 statistical analysis plan; the market fell **68.0%**.

Both follow from the quote-first design working exactly as specified. The prompts anchor the label to a sentence that genuinely appears in the filing and reads like a result — but neither prompt is asked which disclosure in the filing is *material*. Safety events, dose pauses and protocol changes are systematically under-weighted because they are not phrased as endpoint outcomes.

**(c) The label reproduces the sponsor's adjective instead of assessing the evidence.** Q32 Bio's bempikibart Phase 2a filing describes "encouraging clinical activity" and "emerging signals" and is labelled `mixed-positive`; Pyxis Oncology's release of "positive preliminary data" from a seven-patient Phase 1 cohort is labelled `positive`. The documented definition of `positive` is "primary endpoint met with statistical significance, or unambiguous statement of success" — neither filing meets it. Nothing in either prompt asks whether the quoted sentence is a result or a characterisation of one.

**(d) Genuine market pricing.** What survives after (a) to (c) is a real and interpretable asymmetry. Within results announcements, `negative` labels agree with the return 90% of the time and `positive` labels only 56%. Bad news is a surprise; good news is substantially priced in before the filing, and a met endpoint that falls short of what was hoped for still produces a fall. This is not a defect in the data — it is the phenomenon the CFO is asking about, and it is why Question 3 is a question about magnitude rather than direction.

#### Where these patterns come from in the methodology

| Design choice recorded in the methodology | Pattern observed here |
|---|---|
| Round 3 tested B1b, which flagged new results vs restatements; B1a was selected instead | Roughly half of directional records sit in periodic reports and decks; sign agreement 61% vs 46% |
| Quote-first anchoring: the label must be grounded in a verbatim sentence | Labels attach to the most result-like sentence rather than the most material one — safety events and protocol changes are missed (NGNE, SLS above) |
| B3b defaults to `inconclusive` where it cannot populate a statistics field | 4 of 5 sampled `inconclusive` records are trials that have not reported; the category overlaps `no_result` |
| Consensus rule resolves `positive`/`mixed-positive` to `positive` as "same direction" | `mixed-positive` has a mean CAR of −6.6% and a 67% sign-disagreement rate |
| Drug matching between prompts uses substring containment | 455 records (16.0%) name a drug that appears nowhere in the filing; 89 records carry two conflicting labels for the same drug in the same filing |
| Prompts tuned on 10 adversarially selected filings, never re-validated on a random sample; the 37% that is `no_result` represented only through disagreement cases | The stratified sample above is the first random look at that bulk. The sampled `no_result` records are all correct, but restatements inside them are treated differently from restatements elsewhere |
| Statistics extracted but never written to the database | `statistics` empty in all 4,539 rows, `evidence` in all but 45 — the quote-first grounding cannot be audited from this database at all |
| Winsorisation applied to the CAR after cumulating | 54 records (2.1%) pinned at exactly −84.2% or +94.5%; the tails are censored, not measured, and cannot be ranked |

## Q1b: Construct flagging approach (5 marks)

Question 1a produced a set of specific, recurring failure modes. This section turns each of them into a rule that can be evaluated for every one of the 4,539 records, combines them into a single risk score, and uses that score to decide both **what** to review and **in what order**.

Two design decisions are worth stating up front.

**The signals are the Q1a findings, not a generic quality checklist.** Every signal below is traceable to something established in 1a, and the table in the next cell records that link explicitly. Nothing is included because it sounded like a reasonable data-quality check.

**Weights reflect what kind of problem each signal is.** A weight of **3** marks a record that is demonstrably defective on its face — the same drug in the same filing carrying two different labels, or a drug name that appears nowhere in the filing. A weight of **2** marks a record whose label may read perfectly well but which cannot be used as an event study observation, because the filing is not the disclosure event or because the market moved hard against the label. A weight of **1** marks circumstances that raise the probability of error without demonstrating one — single-prompt provenance, a category whose boundary was applied inconsistently, restatement language.

A record is flagged at a score of **3 or more**. That is the point at which either one demonstrated defect, or one event-validity problem plus one risk factor, has accumulated — anything below it is a single soft signal, which on the 1a evidence is not enough to justify an analyst's time.

In [ ]:
# ---------------------------------------------------------------------------
# Q1b: build the flagging signals. Each signal below is one of the failure
# modes established in Q1a, expressed as a rule that can be evaluated for every
# one of the 4,539 records. Weights are 3 for a defect that makes the record
# wrong on its face, 2 for one that makes the record unusable for the event
# study even if the label reads correctly, and 1 for a circumstance that raises
# the prior on error without demonstrating one.
# ---------------------------------------------------------------------------
flag = consensus.merge(filing_txt[["accession", "filing_kind", "n_prior_refs"]],
                       on="accession", how="left") \
                .merge(car_by_row, on=["accession", "drug_name"], how="left")

# Weight 3 - a demonstrated defect in the record itself
flag["S1_conflicting_duplicate"] = (flag.dup_status == "repeated, CONFLICTING labels").astype(int)
flag["S2_drug_absent"]           = (flag.drug_in_text == False).astype(int)
flag["S3_prompt_disagreement"]   = (flag.consensus_type == "flagged_disagreement").astype(int)
flag["S4_missed_result"]         = ((flag.outcome == "no_result") &
                                    (flag.filing_kind == "results announcement")).astype(int)
# Weight 2 - the label may read correctly but the event it is attached to is wrong
flag["S5_not_an_announcement"]   = ((flag.outcome != "no_result") &
                                    (flag.filing_kind != "results announcement")).astype(int)
flag["S6_return_contradiction"]  = ((flag.outcome.map(EXPECTED).notna()) &
                                    (np.sign(flag.car_3day) != flag.outcome.map(EXPECTED)) &
                                    (flag.car_3day.abs() >= 0.10)).astype(int)
# Weight 1 - elevated risk
flag["S7_repeated_record"]       = (flag.dup_status == "repeated, same label").astype(int)
flag["S8_restatement_language"]  = ((flag.outcome != "no_result") & (flag.n_prior_refs >= 3)).astype(int)
flag["S9_single_prompt"]         = flag.consensus_type.isin(["b1a_only", "b3b_only"]).astype(int)
flag["S10_stats_override"]       = flag.consensus_type.isin(
    ["b3b_inconclusive_override", "b1a_inconclusive_override"]).astype(int)
flag["S11_boundary_category"]    = flag.outcome.isin(
    ["mixed-positive", "mixed-negative", "inconclusive"]).astype(int)

WEIGHTS = {"S1_conflicting_duplicate": 3, "S2_drug_absent": 3, "S3_prompt_disagreement": 3,
           "S4_missed_result": 3, "S5_not_an_announcement": 2, "S6_return_contradiction": 2,
           "S7_repeated_record": 1, "S8_restatement_language": 1, "S9_single_prompt": 1,
           "S10_stats_override": 1, "S11_boundary_category": 1}
SIGNAL_SOURCE = {
    "S1_conflicting_duplicate": "Q1a(c): 89 records where one drug in one filing carries two different labels",
    "S2_drug_absent":           "Q1a(c): 455 records name a drug that appears nowhere in the filing",
    "S3_prompt_disagreement":   "Methodology: the 45 records the pipeline itself could not resolve",
    "S4_missed_result":         "Q1a: no_result is the untested 37%; a results headline labelled no_result is a candidate miss",
    "S5_not_an_announcement":   "Q1a(2): sign agreement 61% in announcements vs 46% in periodic reports",
    "S6_return_contradiction":  "Q1a(2): large moves against the label are the ones that distort Q3",
    "S7_repeated_record":       "Q1a(c): 170 records are exact repeats and double-count in any fit",
    "S8_restatement_language":  "Q1a(2): agreement falls 57% -> 38% as restatement phrases rise",
    "S9_single_prompt":         "Methodology: ~969 single-prompt records have no second reading",
    "S10_stats_override":       "Methodology: 373 labels rest on B1a alone after B3b defaulted to inconclusive",
    "S11_boundary_category":    "Q1a(1): the mixed/inconclusive boundaries were applied inconsistently",
}
flag["risk_score"] = sum(flag[k] * w for k, w in WEIGHTS.items())

print("Signals, their weight, and the Q1a finding each comes from\n")
display(pd.DataFrame({"weight": WEIGHTS, "records_triggering": flag[list(WEIGHTS)].sum(),
                      "from": SIGNAL_SOURCE}).sort_values(["weight", "records_triggering"],
                                                          ascending=[False, False]))

In [ ]:
# ---------------------------------------------------------------------------
# Choose the threshold, apply it to all 4,539 classifications, and report.
# ---------------------------------------------------------------------------
calib = pd.DataFrame({"threshold": range(1, 6)})
calib["flagged"] = [int((flag.risk_score >= t).sum()) for t in calib.threshold]
calib["proportion"] = (calib.flagged / len(flag)).round(3)
display(calib)

THRESHOLD = 3     # see the note below the output for why
flag["flagged"] = (flag.risk_score >= THRESHOLD).astype(int)

print("Flagged %d of %d classifications (%.1f%%) at risk_score >= %d\n"
      % (flag.flagged.sum(), len(flag), 100 * flag.flagged.mean(), THRESHOLD))

by_cat = flag.groupby("outcome").agg(classifications=("flagged", "size"),
                                     flagged=("flagged", "sum"))
by_cat["proportion_flagged"] = (by_cat.flagged / by_cat.classifications).round(3)
by_cat["documented_category"] = by_cat.index.isin(SIX)
display(by_cat.sort_values("classifications", ascending=False))

fig, ax = plt.subplots(figsize=(7, 3.5))
sub = by_cat[by_cat.documented_category].sort_values("proportion_flagged")
ax.barh(sub.index, 100 * sub.proportion_flagged, color="#4c72b0")
ax.set_xlabel("% of classifications flagged for review")
ax.set_title("Flagging rate by outcome category")
plt.tight_layout(); plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Ordering the review queue. Flagging says which records are doubtful; it does
# not say which are worth an analyst's time. A wrong label on a filing that
# moved the stock 60% distorts the Question 3 regression far more than a wrong
# label on one that moved it 0.5%, and a record with no event study attached to
# it does not enter Questions 2 or 3 at all. Priority therefore combines how
# likely the record is to be wrong with how much the downstream work depends on
# it being right.
# ---------------------------------------------------------------------------
flag["downstream_leverage"] = flag.car_3day.abs().rank(pct=True).fillna(0.0)
flag["review_priority"] = flag.risk_score * (1 + flag.downstream_leverage)

queue = flag[flag.flagged == 1].nlargest(20, "review_priority")
print("Top of the manual-review queue (20 of %d flagged)\n" % int(flag.flagged.sum()))
display(queue[["accession", "ticker", "filing_date", "drug_name", "outcome", "consensus_type",
               "car_3day", "filing_kind", "risk_score", "review_priority"]].round(3))

# Tiers are sized by what a reviewer can actually do, not by an arbitrary score cut:
# the top 50 records are worth reading in full, the next 250 spot-checking, and the
# remainder are better handled by excluding them from the sensitive analyses.
flag["review_tier"] = pd.Series(pd.NA, index=flag.index, dtype="object")
ranked = flag[flag.flagged == 1].sort_values("review_priority", ascending=False).index
flag.loc[ranked[:50], "review_tier"] = "tier 1 - read in full"
flag.loc[ranked[50:300], "review_tier"] = "tier 2 - spot check"
flag.loc[ranked[300:], "review_tier"] = "tier 3 - exclude or screen"
print("Review effort by tier:")
display(flag.review_tier.value_counts().sort_index().to_frame("records"))
print("Flagged records carrying no CAR, so never entering Q2/Q3: %d of %d"
      % (int(((flag.flagged == 1) & flag.car_3day.isna()).sum()), int(flag.flagged.sum())))

#### What the flagging shows

**2,015 of 4,539 classifications (44.4%) are flagged for review.** The breakdown by category is the important output, because it says which parts of the dataset the IR team can lean on and which they cannot:

| Category | Records | Flagged | Proportion |
|---|---|---|---|
| `no_result` | 1,694 | 226 | 13.3% |
| `negative` | 403 | 155 | **38.5%** |
| `positive` | 1,898 | 1,157 | 61.0% |
| `mixed-negative` | 85 | 71 | 83.5% |
| `inconclusive` | 209 | 177 | 84.7% |
| `mixed-positive` | 205 | 184 | **89.8%** |
| 7 undocumented composite labels | 45 | 45 | 100% |

The ordering matches the Q1a findings closely, which is the main check that the score behaves sensibly rather than just producing a number. Among the directional categories, `negative` — the one shown above to agree with the market 90% of the time in a genuine announcement — is by some way the least flagged. The two mixed categories, whose boundary was shown to follow the sponsor's framing rather than the endpoint, are flagged at close to nine in ten, as is `inconclusive`, which was shown to be functioning as a second `no_result`.

**A 44% flag rate needs interpreting, not just reporting.** It does not mean 44% of the labels are wrong. The largest single contributor by far is event validity: 2,128 records sit in filings that are not results announcements, and in most of those the label is *right about the trial* while the abnormal return next to it is measuring a quarterly earnings reaction. For Question 3 that is the more damaging of the two problems, because a wrong label adds noise whereas a mis-dated event adds noise correlated with document type — and document type is itself correlated with sponsor size and with the size of the move. The signals that indicate the label itself is wrong (conflicting duplicates, absent drug names, unresolved prompt disagreements) reach 589 records, or 13% of the table. That is the number to quote if the question is "how often is the classification wrong", as distinct from "how often is this row unusable".

**`no_result` is flagged at only 13.3%, and that is a limitation rather than a clean bill of health.** These records carry no directional claim and are excluded from the event study, so most signals cannot fire on them: there is no drug name to check, no return to contradict, no boundary to misapply. The one signal that reaches them — a filing whose headline announces results but which was classified as containing none — picks up 226 candidates. The methodology is candid that this 37% of the corpus is the least validated part of the pipeline, and nothing here changes that. It is where I would spend manual review effort if the IR team needed confidence that no material readout had been dropped.

**Ordering the queue.** Flagging alone puts 2,015 records in front of an analyst with no indication of which matter. Priority multiplies the risk score by downstream leverage — the percentile rank of |CAR| — because a mislabelled event that moved the stock 60% distorts the Question 3 fit far more than one that moved it half a per cent, and flagged records with no CAR at all never enter Questions 2 or 3. Tiers are then sized by what a reviewer can realistically do rather than by a score cut: 50 records to read in full, 250 to spot-check, the remainder to be handled by exclusion.

The resulting tier 1 is a good check on the whole approach. It is dominated by `mixed-positive` labels sitting in periodic reports and investor decks alongside falls of 15–70% — Neurogene's NGN-401, Lyra's LYR-210, Ocular's OTX-TP, Summit's ivonescimab — plus the unresolved prompt disagreements. That is precisely the intersection of failure modes Q1a identified, arrived at mechanically rather than by picking cases that suited the story.

**What I would do with this in practice.** Read tier 1 by hand. For Question 3, treat the event-validity signal as an exclusion rather than a review item: restricting the modelling set to results announcements costs sample size but removes the mis-dated observations wholesale, and Question 3b is the place to measure what that trade actually buys. For Question 2, which is descriptive and does not partition, the flagged records can stay in — a mis-dated event still describes a real trial, and it is the trial's characteristics, not the timing of its disclosure, that drive the clustering.

## Q1c: Clean and vectorise the data (5 marks)

The brief says cleaning is judged on whether it serves the uses downstream, not on how exhaustive it is. The uses are two, and they place different demands on the features:

- **Question 2** clusters trials to find comparables for the three Asclepius readouts. It is descriptive, uses the whole dataset and does not partition. What it needs from the text is *topic*: what disease, what modality, what kind of study.
- **Question 3** predicts signed CAR under cross-validation. What it needs from the text is *polarity*: whether the result was good or bad, and how emphatically. And every step that learns from the data has to be fitted inside the fold.

So this section does four things: look at the text before deciding what to do to it; clean the structured fields the two questions actually use; establish which columns cannot be used at all because they are not observable at the filing date; and build a text representation, then test whether it is fit for both uses rather than assuming it.

In [ ]:
# ---------------------------------------------------------------------------
# Q1c step 1: look at the text before deciding what to do to it. The filing text
# was extracted programmatically and handed over uncleaned, so the question is
# what is actually in it that a model should not see.
# ---------------------------------------------------------------------------
raw = filing_txt.filing_text
print("Filing text, raw: %d filings, %d - %d characters (median %d)"
      % (len(raw), raw.str.len().min(), raw.str.len().max(), raw.str.len().median()))
print("\nFirst 600 characters of one filing, verbatim:\n")
print(textwrap.fill(raw.iloc[0][:600], 98))

probe = {
    "SEC cover page ('Check the appropriate box')": r"Check the appropriate box below",
    "Safe-harbour / forward-looking block":         r"forward-looking statements",
    "Signature page":                               r"Pursuant to the requirements of the Securities",
    "Smart quotes / bullet glyphs":                 r"[\u2018\u2019\u201c\u201d\u25aa\u2022\u00b7]",
    "Slide-deck spaced hyphens ('well - tolerated')": r"[a-z]{3,} - [a-z]{3,}",
}
print("\nHow widespread each issue is:")
display(pd.DataFrame({"filings_affected": {k: int(raw.str.contains(v, regex=True, case=False).sum())
                                           for k, v in probe.items()}})
        .assign(pct=lambda d: (100 * d.filings_affected / len(raw)).round(1)))

print("\nEffect of clean_filing_text() defined in Q1a:")
display(pd.DataFrame({"raw": raw.str.len().describe(), "cleaned": filing_txt.clean.str.len().describe()})
        .round(0))

#### What the text needed, and what was deliberately left alone

The filing text is raw EDGAR output and carries three kinds of material that would dominate any vectorisation while saying nothing about the trial:

- the **SEC cover page**, which is close to identical across all 3,794 filings;
- the **safe-harbour block**, which is long, formulaic, and — importantly — full of exactly the vocabulary a result classifier would key on ("futility analyses", "interim results", "the success, timing and cost of our ongoing clinical trials");
- the **exhibit index and signature page**.

Together these are 43.7% of all characters. Removing them takes the median filing from 19,186 to 7,986 characters. The safe-harbour block is the one that matters most: it is the reason an uncleaned embedding of two filings with opposite outcomes still looks similar.

There is also character-level damage from the extraction — smart quotes, bullet glyphs arriving as stray letters, and slide decks that emit spaced hyphens, so "well-tolerated" arrives as "well - tolerated" and would tokenise as three tokens. These are normalised.

**What was deliberately left alone.** Numbers, p-values and units are kept: "p=0.009" and "did not meet its primary endpoint" are the sentences the whole analysis turns on, and a conventional clean that strips digits and punctuation would destroy them. No stemming or stopword removal is applied to the text passed to the sentence encoder, because the encoder is trained on ordinary prose and negation is precisely what has to survive. The `indication` and `phase` free-text fields are normalised for modelling but the originals are retained, because Question 2's manual validation needs the human-readable version.

**On the analysis unit.** Questions 2 and 3 work at the level of one classified trial result with an observed market reaction, so cleaning is done on `event_study_enriched` rather than on `consensus_outcomes`. The duplicate records found in Q1a are dropped here; the 1,073 records sitting in multi-drug filings are kept, because each is a distinct trial, but they share one CAR between them and that has to be handled in Question 3 rather than pretended away.

In [ ]:
# ---------------------------------------------------------------------------
# Q1c step 2: the structured fields. The analysis unit for Questions 2 and 3 is
# one classified trial result with an observed market reaction, so cleaning is
# done on event_study_enriched. Only the problems that matter downstream are
# fixed: free-text phase, numeric fields stored as text, the duplicate records
# found in Q1a, and the fields that cannot be used because they are not
# observable at the filing date.
# ---------------------------------------------------------------------------
enriched = pd.read_sql("SELECT * FROM event_study_enriched", conn)
print("event_study_enriched: %d rows, %d filings" % (len(enriched), enriched.accession.nunique()))

# (i) phase is free text: 'Phase 3', 'Phase III', 'Phase 2b', 'Pivotal', ...
print("\nDistinct raw phase strings: %d. phase_clean already collapses most of them:"
      % enriched.phase.nunique())
display(enriched.phase_clean.value_counts(dropna=False).to_frame("rows").head(8))

def phase_bucket(p):
    s = str(p).lower()
    if "4" in s or "post-approval" in s or "post-marketing" in s: return "Phase 4"
    if re.search(r"\b3\b|iii|pivotal", s): return "Phase 3"
    if re.search(r"\b2\b|\bii\b|2a|2b", s): return "Phase 2"
    if re.search(r"\b1\b|\bi\b|1a|1b", s): return "Phase 1"
    return "unknown"
enriched["phase_bucket"] = enriched.phase.map(phase_bucket)
display(enriched.phase_bucket.value_counts().to_frame("rows"))

# (ii) numeric fields stored as TEXT
for col in ["enrollment", "n_arm_groups", "n_interventions", "n_primary_outcomes",
            "n_secondary_outcomes", "n_locations"]:
    enriched[col + "_num"] = pd.to_numeric(enriched[col], errors="coerce")
print("\nNumeric conversion of text columns (count of usable values / %d rows):" % len(enriched))
display(enriched[[c + "_num" for c in ["enrollment", "n_arm_groups", "n_locations"]]]
        .describe().round(1))

# (iii) duplicates carried through from consensus_outcomes
before = len(enriched)
enriched = enriched.drop_duplicates(subset=["accession", "drug_name"], keep="first")
print("\nDropped %d exact (filing, drug) duplicates -> %d analysis rows" % (before - len(enriched), len(enriched)))

# (iv) linkage quality: disclosure_lag_days is negative for a quarter of the rows,
#      meaning the linked study's primary completion post-dates the 8-K.
print("\ndisclosure_lag_days (days from primary completion to filing):")
display(enriched.disclosure_lag_days.describe().round(0).to_frame())
display(enriched.lag_quality.value_counts(dropna=False).to_frame("rows"))
print("Rows with a negative lag (study completed after the filing): %d"
      % int((enriched.disclosure_lag_days < 0).sum()))

In [ ]:
# ---------------------------------------------------------------------------
# Q1c step 3: which columns cannot be used, because they are not observable
# before the market reaction they would be used to predict. ClinicalTrials.gov
# and Drugs@FDA were scraped once, in 2026, so every status-like field holds
# today's value, not the value as at the filing date. An 8-K filed in 2012 whose
# drug was approved in 2019 carries fda_approved = 1 in this table.
# ---------------------------------------------------------------------------
BLOCKED = {
    "ar_day0":            "a component of car_3day itself",
    "car_3day_raw":       "the same event, before winsorisation",
    "alpha":              "market-model parameters fitted around this event",
    "beta":               "market-model parameters fitted around this event",
    "n_evt_days":         "counts days inside the event window",
    "overall_status":     "current trial status, not status at the filing date",
    "completion_date":    "may post-date the filing",
    "primary_completion": "revised after the fact; drives disclosure_lag_days",
    "fda_approved":       "current approval status; often granted years after the filing",
    "fda_approval_date":  "same",
    "results_first_posted": "posted after the filing",
    "has_results":        "same",
    "last_update_posted": "a 2026 snapshot",
}
present = [c for c in BLOCKED if c in enriched.columns]
display(pd.DataFrame({"reason_excluded": {c: BLOCKED[c] for c in present}}))

# How much damage would including one of these do? overall_status is the clearest
# case: it is the trial's status as scraped in 2026, and a trial that was stopped
# because the drug failed reads as TERMINATED forever afterwards.
scored = enriched.dropna(subset=["car_3day"])
print()
print("Mean CAR by (leaky) *current* trial status - none of which was knowable on the filing date:")
display(scored.groupby("overall_status").car_3day.agg(["size", "mean", "median"])
              .round(3).sort_values("size", ascending=False).head(6))

# fda_approved is stored as the strings 'True'/'False', which silently becomes all-NaN
# under pd.to_numeric - worth noting as a trap in its own right.
fda = scored.fda_approved.map({"True": 1, "False": 0})
print("Mean CAR by (leaky) current FDA-approval status of the drug:")
display(scored.assign(fda=fda).dropna(subset=["fda"])
              .groupby("fda").car_3day.agg(["size", "mean", "median"]).round(3))

SAFE_STRUCTURED = ["phase_bucket", "enrollment_num", "n_arm_groups_num", "n_interventions_num",
                   "n_primary_outcomes_num", "n_secondary_outcomes_num", "n_locations_num",
                   "allocation", "masking", "intervention_model", "primary_purpose",
                   "lead_sponsor_class", "has_dmc", "market_cap"]
print("\nStructured features carried forward to Questions 2 and 3:")
print(", ".join(SAFE_STRUCTURED))
print("\nmarket_cap is retained because it varies within ticker (%d of %d tickers have more "
      "than one value), which is consistent with a point-in-time measure rather than a 2026 snapshot."
      % ((enriched.dropna(subset=["market_cap"]).groupby("ticker").market_cap.nunique() > 1).sum(),
         enriched.dropna(subset=["market_cap"]).ticker.nunique()))

#### Vectorising the filing text

Three decisions matter here and are tested rather than assumed in the cell after next: which text to embed, how much of it, and how far to reduce it.

In [ ]:
# ---------------------------------------------------------------------------
# Q1c step 4: vectorise the filing text with sentence embeddings.
#
# Embedding a whole 8-K is not useful: after cleaning, the median filing is still
# ~8,000 characters, most of it commercial and financial narrative about other
# products. What Questions 2 and 3 need is the part of the filing that describes
# *this* trial's result. Each record is therefore reduced to the sentences that
# mention the classified drug and carry result language, using the same
# result_sentences() helper the Q1a reading was done with, and those sentences
# are embedded.
#
# The model is loaded from a local copy if one is present so the notebook runs
# without network access; otherwise it is fetched from the hub.
# ---------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer

MODEL_DIR, MODEL_HUB = "models/all-MiniLM-L6-v2", "sentence-transformers/all-MiniLM-L6-v2"
encoder = SentenceTransformer(MODEL_DIR if os.path.isdir(MODEL_DIR) else MODEL_HUB)

clean_lookup = filing_txt.set_index("accession")["clean"]

def embed_records(df, k=3):
    """Mean of the embeddings of the k most result-relevant sentences per record."""
    per_row = [result_sentences(clean_lookup.get(a, ""), d, k=k, hi=400)[:k]
               for a, d in zip(df.accession, df.drug_name)]
    flat = [s for sents in per_row for s in sents]
    vecs = encoder.encode(flat, batch_size=128, normalize_embeddings=True, show_progress_bar=False)
    out, pos = [], 0
    for sents in per_row:
        out.append(vecs[pos:pos + len(sents)].mean(axis=0)); pos += len(sents)
    E = np.vstack(out)
    return E / np.linalg.norm(E, axis=1, keepdims=True)

print("Encoder ready: %s, %d output dimensions."
      % (os.path.basename(MODEL_DIR), encoder.get_sentence_embedding_dimension()))
print("Records to encode: %d. The next cell runs the encoder three times to compare "
      "sentence counts, which takes a few minutes on CPU." % len(enriched))

In [ ]:
# ---------------------------------------------------------------------------
# Q1c step 5: is the vectorisation fit for what it will be used for?
#
# The test used here is whether the representation can recover the one property
# of the filing that is known independently - whether the result was positive or
# negative - under cross-validation. This is a diagnostic on the features, not a
# model for Question 3; the target there is signed CAR, not the label.
#
# Three things are compared: the length of text embedded, the number of
# components retained, and a TF-IDF baseline on the same sentences.
# ---------------------------------------------------------------------------
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

polar = enriched.outcome.isin(["positive", "negative"]).values
y_polar = (enriched.outcome[polar] == "negative").astype(int).values
print("Diagnostic subset: %d records (%.0f%% negative)" % (polar.sum(), 100 * y_polar.mean()))

rows = []
for k in [1, 3, 10]:
    E = embed_records(enriched, k=k)
    auc_full = cross_val_score(LogisticRegression(max_iter=3000), E[polar], y_polar,
                               cv=5, scoring="roc_auc").mean()
    auc_pca = cross_val_score(make_pipeline(PCA(n_components=50, random_state=MEMBER_ID),
                                            LogisticRegression(max_iter=3000)),
                              E[polar], y_polar, cv=5, scoring="roc_auc").mean()
    rows.append({"sentences_embedded": k, "AUC_384_dims": round(auc_full, 3),
                 "AUC_50_components": round(auc_pca, 3)})
    if k == 3:
        emb = E
        sent_text = [" ".join(result_sentences(clean_lookup.get(a, ""), d, k=3, hi=400)[:3])
                     for a, d in zip(enriched.accession, enriched.drug_name)]

tfidf_auc = cross_val_score(
    make_pipeline(TfidfVectorizer(min_df=5, max_features=20000, stop_words="english",
                                  ngram_range=(1, 2)), LogisticRegression(max_iter=3000)),
    pd.Series(sent_text)[polar], y_polar, cv=5, scoring="roc_auc").mean()

display(pd.DataFrame(rows))
print("TF-IDF on the same three sentences, for comparison: AUC %.3f" % tfidf_auc)

In [ ]:
# ---------------------------------------------------------------------------
# Q1c step 6: dimensionality reduction, and the difference between the two uses.
#
# Question 2 clusters the whole dataset and does not partition it, so the
# reduction is fitted once on everything - that is the intended behaviour for a
# descriptive analysis. Question 3 cross-validates, and a PCA fitted on all rows
# has already seen the held-out fold. The reduction is therefore refitted inside
# each fold there, which is what a Pipeline does. Both are shown below so the
# size of the difference is on the record rather than assumed.
# ---------------------------------------------------------------------------
pca_full = PCA(n_components=50, random_state=MEMBER_ID).fit(emb)
Z = pca_full.transform(emb)
ev = pca_full.explained_variance_ratio_
print("50 components retain %.1f%% of embedding variance" % (100 * ev.sum()))
display(pd.DataFrame({"components": [10, 20, 30, 50],
                      "cumulative_variance": [round(ev[:k].sum(), 3) for k in [10, 20, 30, 50]]}))

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(range(1, 51), np.cumsum(ev), marker=".")
ax.axhline(ev[:50].sum(), ls="--", lw=0.8, color="grey")
ax.set_xlabel("components"); ax.set_ylabel("cumulative explained variance")
ax.set_title("PCA on the filing-text embeddings"); plt.tight_layout(); plt.show()

# Leaked vs honest evaluation, on the same folds.
leaked = cross_val_score(LogisticRegression(max_iter=3000), Z[polar], y_polar,
                         cv=5, scoring="roc_auc").mean()
honest = cross_val_score(make_pipeline(PCA(n_components=50, random_state=MEMBER_ID),
                                       LogisticRegression(max_iter=3000)),
                         emb[polar], y_polar, cv=5, scoring="roc_auc").mean()
print("\nPCA fitted on all rows, then cross-validated (leaks): AUC %.3f" % leaked)
print("PCA refitted inside each fold (honest):               AUC %.3f" % honest)
print("Difference: %+.3f" % (leaked - honest))

# The artefacts Questions 2 and 3 will use.
TEXT_PCS = pd.DataFrame(Z, columns=["txt_pc%02d" % (i + 1) for i in range(Z.shape[1])],
                        index=enriched.index)
analysis = pd.concat([enriched.reset_index(drop=True),
                      TEXT_PCS.reset_index(drop=True)], axis=1)
# consensus_outcomes holds duplicate (filing, drug) keys, so the flags are collapsed
# to one row per key - taking the worst score - before joining, to avoid the merge
# silently expanding the analysis table.
flags_by_key = (flag.groupby(["accession", "drug_name"], as_index=False)
                    .agg(risk_score_q1b=("risk_score", "max"), flagged_q1b=("flagged", "max")))
analysis = analysis.merge(flags_by_key, on=["accession", "drug_name"], how="left")
analysis[["risk_score_q1b", "flagged_q1b"]] = analysis[["risk_score_q1b", "flagged_q1b"]].fillna(0)
analysis["result_sentences"] = sent_text
print("\nAnalysis table for Questions 2-4: %d rows x %d columns "
      "(%d text components, %d flagged by Q1b)"
      % (analysis.shape[0], analysis.shape[1], Z.shape[1], int(analysis.flagged_q1b.sum())))

#### Is the vectorisation fit for purpose? Testing rather than assuming

The diagnostic above asks whether the representation can recover, under cross-validation, a property of the filing that is known independently: whether the classified outcome was `positive` or `negative`. This is not the Question 3 model — the target there is signed CAR — it is a check that the features carry the information the downstream work will ask them for.

Three results come out of it, and the third changed what I used.

**Embedding fewer, better-chosen sentences beats embedding more text.** Mean-pooling the three most result-relevant sentences outperforms a single sentence, and adding more sentences past that adds nothing. Sentence embeddings average away contrasts, so the more unrelated material is folded into one vector, the more the polarity of the result is diluted by the surrounding commercial narrative — which in a periodic report is most of the document. This is also why the drug-targeted sentence selection matters: in a multi-drug filing, embedding the filing as a whole gives every drug in it the same vector.

**Dimensionality reduction costs some signal, and the cost is worth knowing.** Reducing 384 dimensions to 50 loses a few points of AUC while retaining about 63% of embedding variance. That trade is worth making for clustering, where 384 dimensions make Euclidean distance nearly meaningless and the reduction is doing real work; it is a genuine cost in Question 3, which is one reason the text features there are not limited to the reduced embedding.

**The embedding is a topic model, not a sentiment model — and that determined the final design.** TF-IDF on exactly the same three sentences reaches a materially higher AUC than the embedding does. The reason is visible once stated: the signal separating a good result from a bad one is carried by a handful of literal phrases — "did not meet", "met its primary endpoint", "discontinued", "statistically significant" — and TF-IDF represents those directly, while a 384-dimension sentence embedding compresses them into a space it is also using for disease area, modality and study design. The embedding is genuinely good at what Question 2 needs (grouping trials by what they are about) and comparatively weak at what Question 3 needs (how the result went).

I therefore carry **both** representations forward rather than choosing between them: the reduced embedding as the text contribution to the clustering in Question 2 and as topical features in Question 3, and TF-IDF as a separate representation built inside the Question 3 cross-validation. That also satisfies Question 3a's requirement for at least two distinct text representations, and it is the conclusion the numbers forced rather than a convenient one — my first attempt used the embedding alone, on a longer digest, and scored worse than a bag of words.

#### Leakage, and how the two uses differ

**Feature-level leakage.** The columns excluded above are excluded because ClinicalTrials.gov and Drugs@FDA were scraped once, in 2026, so every status-like field holds today's value rather than the value as at the filing date. `overall_status` is the clearest case: trials recorded as TERMINATED average a −12.2% CAR against −2.1% for COMPLETED and +1.2% for ACTIVE_NOT_RECRUITING. A model given that column would look impressively predictive while doing nothing but reading the trial's obituary, written years after the filing. `fda_approved` is the same trap in milder form, and comes with a second one: it is stored as the strings `'True'`/`'False'`, so `pd.to_numeric` turns the whole column silently into NaN rather than failing. `ar_day0` and `car_3day_raw` are more obvious, being components of the target itself. `disclosure_lag_days` is retained but with care: it is derived from `primary_completion`, which is revised after the fact, and it is negative for a quarter of the rows, meaning the linked study's recorded completion post-dates the filing. That is a linkage-quality signal as much as a feature.

One caveat I cannot engineer away and so record instead: the event window is [−1, +1], so it includes the trading day *before* the filing. Strictly, no feature derived from the filing text is observable at the start of the window. That is a property of the event study design supplied with the dataset rather than a choice available to me, and it biases every text feature in the same direction — towards looking slightly better than it is. It is one reason I would not present Question 3's performance as a floor.

**Pipeline-level leakage, and the Q2/Q3 difference.** This is where the two uses genuinely diverge. In Question 2 the clustering is descriptive and the brief is explicit that the data should not be partitioned, so fitting PCA once on all rows is the correct behaviour: there is no held-out set for it to contaminate, and a reduction refitted per-fold would not even be well defined. In Question 3 a PCA fitted on all rows has already seen the test fold, so the reduction is refitted inside each fold via a `Pipeline`. The cell above measures that difference on the same folds rather than asserting it. The gap turns out to be small, which is what one expects from an unsupervised reduction on 2,400 rows — but small-and-measured is a different claim from assumed-negligible, and the same discipline applied to a supervised step would not be optional.

The sentence encoder itself is pre-trained and never sees the target, so encoding all rows once is not leakage. The TF-IDF vectoriser *does* learn its vocabulary and document frequencies from the data, so in Question 3 it is fitted inside the fold; in Question 2b, where the brief asks for keywords that describe the clusters, fitting on everything is correct.

**What I satisfied myself of.** That the features carry outcome-relevant signal at all — they do, and I have measured how much rather than inspecting a few examples. That the signal survives the reduction well enough for clustering and imperfectly enough to matter for prediction, which is why both representations are kept. And that nothing carried forward is observable only after the return it would be used to predict.

I expect to return to this cell. The brief invites it, and the most likely revision is a tighter restriction of the Question 3 modelling set to the filings the Q1b flags identify as genuine announcements — the Q1a evidence says that would remove a systematic distortion, and Question 3b is the right place to test what it costs in sample size.

## Q1d: Summary for the IR team (500 words or less, 4 marks)

_Your answer (500 words or less — `word_count()` in the Setup cell will check):_

### Verdict

Partly. The labels are good enough to use, but not as the dataset is set up, and two categories should not be used at all.

**What we checked.** We read 30 classifications against the original filings, covering all six outcome types, then compared every directional label to the share price move that followed across all 2,392 filings holding both.

**The good news.** The labels are not guesswork. Filings labelled negative were followed by an average 19% fall and positive ones by a 4% rise — far too large a gap to be chance. As a reading exercise the tool is accurate: in 24 of our 30 cases the label fairly described the filing.

**The main problem is timing, not accuracy.** Only 6 of our 30 filings were the announcement that broke the news. The rest were quarterly reports and investor decks mentioning a result disclosed weeks or months earlier. The label describes the trial correctly, but the price move beside it reacts to something else, usually earnings. Immunic's August filing restating a June trial failure sits next to a **+42%** move, because that filing was about the quarter.

This holds across the dataset. In genuine announcements the label agrees with the direction of the move 61% of the time; in quarterly reports, 46% — worse than a coin toss. For negative results, 90% against 53%.

**Two categories are not fit for use.** "Mixed-positive" is treated as good news, but those filings were followed by an average **6.6% fall**: the market reads them as disappointments. "Inconclusive" mostly does not mean an unclear result — four of our five cases were trials that had not reported.

**A blind spot.** Where a filing carries both good and bad news, the tool picks up the good. Neurogene's release describing "favourable safety and efficacy data" is labelled positive; the news in that filing was a serious adverse event and a halted dose. The stock fell 68%. Safety problems and protocol changes are what it misses.

**A real finding, not a defect.** Setting timing aside, a genuine pattern remains: bad news moves the price reliably, good news often does not, because success is already partly expected. That bears on ASC-101 and ASC-204. A positive readout is not a dependable catalyst, and planning a raise on the assumption that it is would be optimistic.

**What we did about it.** We built a screen scoring all 4,539 classifications on the problems we found. It flags **44%**, very unevenly: 39% of negative labels, 61% of positive, 84–90% of mixed and inconclusive. Most flags are timing, not wrong labels — the ones saying the label itself is wrong reach 13%. The queue is ranked so the records that would most distort our modelling come first.

**Recommendation.** Restrict the analysis to genuine announcements. Treat "negative" as the most trustworthy label. Do not treat "mixed-positive" as favourable. Read the top of the review queue before any number goes to the board.

### AI use — Question 1

> **TO COMPLETE BEFORE SUBMISSION.** Question 5 and the AI-use guidance require this to be *your own* truthful account. The notes below record what happened in the AI session so you have the facts to work from — rewrite them in your own words, and cut or correct anything that does not match your own involvement. Do not submit this paragraph as it stands.

Claude (Opus 5, via Claude Code) was used across Question 1: to write and debug the cleaning, sampling, flagging and vectorisation code, to extract result-relevant excerpts from the filings so they could be read, and to draft the commentary.

Points in the session where the first output was wrong and was changed:

- The document-type classifier initially read the guidance sentence *"we expect to report top-line results in late fourth quarter 2021"* as a quarterly report, misfiling a genuine Theravance topline announcement. It was rebuilt to give the headline precedence over the body, and then validated against the 30 hand-read filings (28/30 on the announcement / not-announcement split).
- The disagreement cases were first selected by largest |CAR|, which returns ties because `car_3day` is winsorised at the 1st and 99th percentiles. Re-selected stratified by document type instead.
- The drug-presence check was first run against the *cleaned* filing text, which reports a drug as absent when all the cleaning did was remove the safe-harbour block that named it. Moved to the raw text.
- The first vectorisation embedded a long multi-sentence digest and scored below a bag-of-words baseline. That is what led to testing sentence counts, and to keeping TF-IDF as a second representation rather than choosing between them.
- The first flagging pipeline left `no_result` — 37% of the corpus — entirely unflagged. A missed-result signal was added so that part of the data has at least one way of being caught.

Full conversation logs are in `ai_logs/`.

---

# Question 2: Identify comparable trials (20 marks)


## Q2a: Clustering (8 marks)

In [ ]:
# Q2a: Prepare features for clustering

In [ ]:
# Q2a: Clustering algorithm

In [ ]:
# Q2a: Internal validation and comparison of alternatives

## Q2b: Discriminator modelling (4 marks)

In [ ]:
# Q2b: TF-IDF vectorisation of filing text

In [ ]:
# Q2b: Random forest discriminator (default hyperparameters, per-cluster variable importance)

## Q2c: Manual validation (4 marks)

In [ ]:
# Q2c: Manual validation

## Q2d: Summary for the IR team (500 words or less, 4 marks)

_Your answer (500 words or less — `word_count()` in the Setup cell will check):_


### AI use — Question 2

_Note here whether and how you used AI for Question 2, or write 'No AI'. This is a one-line disclosure, not the log itself — save full conversations in `ai_logs/` as described in 'Recording your AI use.docx'. If you called an AI model from your own code above, use `log_ai()` from the Setup cell instead of writing the exchange here by hand._

---

# Question 3: Predict the magnitude of market reactions (27 marks)


## Q3a: Construct feature set (7 marks)

In [ ]:
# Q3a: Load data

In [ ]:
# Q3a: Feature group 1 — Trial and outcome features

In [ ]:
# Q3a: Feature group 2 — Text-based features (at least two representations)

In [ ]:
# Q3a: Feature group 3 — Company/market context features

In [ ]:
# Q3a: Describe feature groups

## Q3b: Construct magnitude prediction models (10 marks)

In [ ]:
# Q3b: Model comparison (at least two model families)

In [ ]:
# Q3b: Ablation analysis (contribution of each feature group)

In [ ]:
# Q3b: Practical discriminatory power (e.g. quintile analysis)

In [ ]:
# Q3b: Data leakage discussion

## Q3c: Asymmetric predictability (6 marks)

In [ ]:
# Q3c: Within-outcome-group analysis

## Q3d: Summary for the IR team (500 words or less, 4 marks)

_Your answer (500 words or less — `word_count()` in the Setup cell will check):_


### AI use — Question 3

_Note here whether and how you used AI for Question 3, or write 'No AI'. This is a one-line disclosure, not the log itself — save full conversations in `ai_logs/` as described in 'Recording your AI use.docx'. If you called an AI model from your own code above, use `log_ai()` from the Setup cell instead of writing the exchange here by hand._

---

# Question 4: Apply your analysis and prepare for deployment (17 marks)


## Q4a: Estimate market reactions for Asclepius's readouts (10 marks)

_Parts (i) to (v) are set out in the brief — work from there, not from these labels. Use the code cells for the work and the markdown cells for what you conclude from it._


**Q4a (i)**

_Your answer:_


In [ ]:
# Q4a (ii): estimates for each readout and scenario


_Your answer:_


In [ ]:
# Q4a (iii): the effect of the features you do not have


_Your answer:_


In [ ]:
# Q4a (iv): workings for the range and its decomposition


**Q4a (iv)**

_Your answer:_


**Q4a (v)**

_Your answer:_


## Q4b: Validate, monitor, and refresh plan (4 marks)

_Written answer — no code required._


_Your answer:_

## Q4c: Advise the IR team on presenting the analysis (300 words or less, 3 marks)

_Your answer (300 words or less — `word_count()` in the Setup cell will check):_


### AI use — Question 4

_Note here whether and how you used AI for Question 4, or write 'No AI'. This is a one-line disclosure, not the log itself — save full conversations in `ai_logs/` as described in 'Recording your AI use.docx'. If you called an AI model from your own code above, use `log_ai()` from the Setup cell instead of writing the exchange here by hand._

---

## Q5: Video presentation (15 marks)

This question is answered as a **video**, not in this notebook — see the brief for what your handover must address, and 'Recording your AI use.docx' for what to submit alongside it.

Before submitting:

- Record your 3–5 minute video and include it with your submission.
- Make sure every dated conversation file is saved under `ai_logs/`, then run the provided `compile_ai_logs.py` script from your submission folder to generate `ai_logs/AI_LOGS.md`. That single file is what you submit.
- If a conversation could not be saved at all, record it honestly with `write_manual_note()` from the Setup cell rather than leaving a silent gap.
- Complete the AI log cell below.
- Paste your YouTube URL into the **Video link** cell at the bottom of this notebook. Without it the marker cannot find your video.

### AI use — Question 5

_Note here whether and how you used AI for Question 5, or write 'No AI'. This is a one-line disclosure, not the log itself — save full conversations in `ai_logs/` as described in 'Recording your AI use.docx'. If you called an AI model from your own code above, use `log_ai()` from the Setup cell instead of writing the exchange here by hand._

---
## Cleanup

In [ ]:
conn.close()
print('Done.')

---

## Video link

The brief requires your YouTube video URL to appear as a hyperlink at the bottom of this notebook. Replace the placeholder below — an unlisted link is fine, a private one is not, because the marker has to be able to open it.

**My video:** [INSERT YOUTUBE URL HERE]
